# ACP-STGAT Motion Predictor

**ACP-STGAT = Action-Conditioned Physics-Informed Spatio-Temporal Graph Attention Transformer**

This notebook trains a stronger Level 2 motion predictor for the AI Martial Arts Coach.

Model idea:

```text
past landmarks
+ velocity
+ acceleration
+ physics prior
+ joint attention
+ action/mistake context
=> future landmarks
```

App-compatible ONNX contract remains:

```text
input  past_landmarks:   [1, 60, 33, 3]
output future_landmarks: [1, 30, 33, 3]
```

The extra physics/action features are computed inside the model from `past_landmarks`, so the current frontend can still call the model with one input.

In [ ]:
!pip -q install torch numpy pandas matplotlib tqdm onnx onnxruntime onnxscript

In [ ]:
import os, json, math, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_JOINTS = 33
COORD_DIM = 3
PAST_FRAMES = 60
FUTURE_FRAMES = 30
MIN_FRAMES = PAST_FRAMES + FUTURE_FRAMES
STRIDE = 5
BATCH_SIZE = 48
MAX_WINDOWS = 50000
USE_SYNTHETIC_FALLBACK = False
DATA_DIR = Path('/content/motion_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('Model: ACP-STGAT')

## Upload Dataset

Supported pose formats:

- `.npz` / `.npy`: `[T,33,3]`, `[N,T,33,3]`, `[T,99]`, `[N,T,99]`
- `.json`: nested MediaPipe landmark data
- `.csv`: long format `frame,joint,x,y,z` or wide format `x_0,y_0,z_0 ...`

Each sequence must contain at least `90` frames by default: 60 past + 30 future.

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, content in uploaded.items():
    target = DATA_DIR / Path(name).name
    target.write_bytes(content)
    print('Saved:', target, len(content), 'bytes')

files_found = sorted([p for p in DATA_DIR.rglob('*') if p.suffix.lower() in {'.npz', '.npy', '.json', '.csv'}])
print('Dataset files found:', len(files_found))
for p in files_found[:20]:
    print(' ', p.name, p.stat().st_size, 'bytes')

In [ ]:
def walk_arrays(obj):
    if isinstance(obj, dict):
        for value in obj.values():
            yield from walk_arrays(value)
    elif isinstance(obj, (list, tuple)):
        for value in obj:
            yield from walk_arrays(value)
    elif isinstance(obj, np.ndarray):
        yield obj


def safe_array(value):
    try:
        arr = np.asarray(value, dtype=np.float32)
        if arr.size == 0:
            return None
        return arr
    except Exception:
        return None


def adapt_joint_count(seq, target_joints=33):
    seq = np.asarray(seq, dtype=np.float32)
    if seq.ndim != 3:
        raise ValueError(f'expected [T,J,C], got {seq.shape}')
    if seq.shape[-1] < 2:
        raise ValueError(f'coordinate dimension too small: {seq.shape}')
    if seq.shape[-1] < 3:
        pad = np.zeros(seq.shape[:-1] + (3 - seq.shape[-1],), dtype=np.float32)
        seq = np.concatenate([seq, pad], axis=-1)
    seq = seq[..., :3]
    joints = seq.shape[1]
    if joints == target_joints:
        return seq
    if joints > target_joints:
        idx = np.linspace(0, joints - 1, target_joints).round().astype(int)
        return seq[:, idx, :]
    pad = np.repeat(seq[:, -1:, :], target_joints - joints, axis=1)
    return np.concatenate([seq, pad], axis=1)


def normalize_mediapipe33(seq):
    seq = adapt_joint_count(seq, NUM_JOINTS)
    left_hip, right_hip = seq[:, 23:24, :], seq[:, 24:25, :]
    root = (left_hip + right_hip) / 2.0
    shoulder_width = np.linalg.norm(seq[:, 11, :] - seq[:, 12, :], axis=-1, keepdims=True)
    shoulder_center = (seq[:, 11, :] + seq[:, 12, :]) / 2.0
    torso = np.linalg.norm(shoulder_center - root[:, 0, :], axis=-1, keepdims=True)
    scale = np.maximum(np.maximum(shoulder_width, torso), 1e-3)
    norm = (seq - root) / scale[:, None, :]
    return np.nan_to_num(norm, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def coerce_sequence_array(arr):
    arr = safe_array(arr)
    if arr is None:
        return []
    results = []
    if arr.ndim == 2 and arr.shape[1] >= NUM_JOINTS * 2:
        channels = 3 if arr.shape[1] >= NUM_JOINTS * 3 else 2
        usable = NUM_JOINTS * channels
        results.append(arr[:, :usable].reshape(arr.shape[0], NUM_JOINTS, channels))
    elif arr.ndim == 3:
        if arr.shape[-1] >= 2:
            results.append(arr)
        elif arr.shape[-1] >= NUM_JOINTS * 2:
            channels = 3 if arr.shape[-1] >= NUM_JOINTS * 3 else 2
            usable = NUM_JOINTS * channels
            for clip in arr:
                results.append(clip[:, :usable].reshape(clip.shape[0], NUM_JOINTS, channels))
    elif arr.ndim == 4:
        for clip in arr:
            if clip.ndim == 3 and clip.shape[-1] >= 2:
                results.append(clip)
    return results


def load_np_file(path):
    sequences = []
    if path.suffix.lower() == '.npy':
        return coerce_sequence_array(np.load(path, allow_pickle=True))
    data = np.load(path, allow_pickle=True)
    for key in data.keys():
        obj = data[key]
        if isinstance(obj, np.ndarray) and obj.dtype == object:
            candidates = []
            try:
                candidates.extend(list(walk_arrays(obj.item())))
            except Exception:
                candidates.extend(list(walk_arrays(obj.tolist())))
        else:
            candidates = [obj]
        for candidate in candidates:
            sequences.extend(coerce_sequence_array(candidate))
    return sequences


def extract_json_landmarks(obj):
    if isinstance(obj, dict):
        for key in ['landmarks', 'pose_landmarks', 'poseLandmarks', 'keypoints', 'joints']:
            if key in obj:
                return extract_json_landmarks(obj[key])
        return None
    if isinstance(obj, list):
        points = []
        for item in obj:
            if isinstance(item, dict) and ('x' in item and 'y' in item):
                points.append([item.get('x', 0.0), item.get('y', 0.0), item.get('z', 0.0)])
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                points.append([item[0], item[1], item[2] if len(item) > 2 else 0.0])
        if points:
            return np.asarray(points, dtype=np.float32)
    return None


def collect_json_sequences(obj):
    sequences = []
    arr = safe_array(obj)
    if arr is not None:
        sequences.extend(coerce_sequence_array(arr))
        if sequences:
            return sequences
    if isinstance(obj, dict):
        for value in obj.values():
            sequences.extend(collect_json_sequences(value))
    elif isinstance(obj, list):
        frames = [extract_json_landmarks(item) for item in obj]
        frames = [f for f in frames if f is not None]
        if len(frames) >= 2:
            max_joints = max(f.shape[0] for f in frames)
            clip = np.zeros((len(frames), max_joints, 3), dtype=np.float32)
            for i, frame in enumerate(frames):
                clip[i, :frame.shape[0], :] = frame[:, :3]
            sequences.append(clip)
        else:
            for value in obj:
                sequences.extend(collect_json_sequences(value))
    return sequences


def load_json_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    return collect_json_sequences(obj)


def find_col(columns, names):
    lower = {c.lower(): c for c in columns}
    for name in names:
        if name in lower:
            return lower[name]
    return None


def load_csv_file(path):
    df = pd.read_csv(path)
    if df.empty:
        return []
    columns = list(df.columns)
    frame_col = find_col(columns, ['frame', 'frame_id', 'frameindex', 'time', 'timestamp', 't'])
    joint_col = find_col(columns, ['joint', 'joint_id', 'landmark', 'landmark_id', 'keypoint', 'keypoint_id'])
    x_col = find_col(columns, ['x', 'x_coord', 'x_coordinate'])
    y_col = find_col(columns, ['y', 'y_coord', 'y_coordinate'])
    z_col = find_col(columns, ['z', 'z_coord', 'z_coordinate'])
    group_col = find_col(columns, ['sequence', 'sequence_id', 'clip', 'clip_id', 'video', 'video_id', 'file', 'sample', 'sample_id'])
    sequences = []

    if frame_col and joint_col and x_col and y_col:
        groups = df.groupby(group_col) if group_col else [(path.stem, df)]
        for _, g in groups:
            frames = sorted(g[frame_col].dropna().unique())
            joints = sorted(g[joint_col].dropna().unique())
            if not len(frames) or not len(joints):
                continue
            frame_index = {v: i for i, v in enumerate(frames)}
            joint_index = {v: int(v) if float(v).is_integer() and 0 <= int(v) < 1000 else i for i, v in enumerate(joints)}
            joint_count = max(joint_index.values()) + 1
            clip = np.zeros((len(frames), joint_count, 3), dtype=np.float32)
            for row in g.itertuples(index=False):
                rd = row._asdict()
                fi = frame_index.get(rd[frame_col])
                ji = joint_index.get(rd[joint_col])
                if fi is None or ji is None:
                    continue
                clip[fi, ji, 0] = float(rd[x_col])
                clip[fi, ji, 1] = float(rd[y_col])
                clip[fi, ji, 2] = float(rd[z_col]) if z_col else 0.0
            sequences.append(clip)
        return sequences

    lower_cols = {c.lower(): c for c in columns}
    joint_triplets = []
    for j in range(100):
        candidates = [
            (f'x_{j}', f'y_{j}', f'z_{j}'),
            (f'{j}_x', f'{j}_y', f'{j}_z'),
            (f'joint_{j}_x', f'joint_{j}_y', f'joint_{j}_z'),
            (f'landmark_{j}_x', f'landmark_{j}_y', f'landmark_{j}_z'),
        ]
        for x_name, y_name, z_name in candidates:
            if x_name in lower_cols and y_name in lower_cols:
                joint_triplets.append((lower_cols[x_name], lower_cols[y_name], lower_cols.get(z_name)))
                break
    if joint_triplets:
        groups = df.groupby(group_col) if group_col else [(path.stem, df)]
        for _, g in groups:
            clip = np.zeros((len(g), len(joint_triplets), 3), dtype=np.float32)
            for ji, (xc, yc, zc) in enumerate(joint_triplets):
                clip[:, ji, 0] = pd.to_numeric(g[xc], errors='coerce').fillna(0).to_numpy(np.float32)
                clip[:, ji, 1] = pd.to_numeric(g[yc], errors='coerce').fillna(0).to_numpy(np.float32)
                if zc:
                    clip[:, ji, 2] = pd.to_numeric(g[zc], errors='coerce').fillna(0).to_numpy(np.float32)
            sequences.append(clip)
    return sequences


def synthetic_mediapipe33(frames=180, motion_type=0, noise=0.01):
    base = np.zeros((33, 3), dtype=np.float32)
    base[23] = [-0.12, 0.0, 0]; base[24] = [0.12, 0.0, 0]
    base[11] = [-0.18, 0.55, 0]; base[12] = [0.18, 0.55, 0]
    base[13] = [-0.38, 0.35, 0]; base[14] = [0.38, 0.35, 0]
    base[15] = [-0.58, 0.18, 0]; base[16] = [0.58, 0.18, 0]
    base[25] = [-0.13, -0.45, 0]; base[26] = [0.13, -0.45, 0]
    base[27] = [-0.15, -0.88, 0]; base[28] = [0.15, -0.88, 0]
    base[0] = [0, 0.85, 0]; base[31] = [-0.17, -0.98, 0]; base[32] = [0.17, -0.98, 0]
    for i in range(33):
        if not np.any(base[i]):
            base[i] = base[0]
    t = np.linspace(0, 1, frames)
    seq = np.repeat(base[None], frames, axis=0).copy()
    amp = random.uniform(0.04, 0.22)
    phase = random.random() * 2 * math.pi
    if motion_type == 0:
        reach = amp * np.clip(np.sin(np.pi * t), 0, None)
        seq[:, 14, 0] += 0.35 * reach; seq[:, 16, 0] += reach
    elif motion_type == 1:
        guard = amp * np.sin(2 * math.pi * 2.0 * t + phase)
        seq[:, 13, 1] += guard; seq[:, 14, 1] -= guard; seq[:, 15, 1] += guard; seq[:, 16, 1] -= guard
    elif motion_type == 2:
        down = amp * (1 - np.cos(2 * math.pi * t))
        seq[:, :, 1] -= down[:, None]
    else:
        sway = amp * np.sin(2 * math.pi * 1.5 * t + phase)
        seq[:, :, 0] += sway[:, None]
    seq += np.random.normal(0, noise, seq.shape).astype(np.float32)
    return normalize_mediapipe33(seq)

In [ ]:
loaders = {'.npz': load_np_file, '.npy': load_np_file, '.json': load_json_file, '.csv': load_csv_file}
raw_sequences = []
skipped = Counter()
files_found = sorted([p for p in DATA_DIR.rglob('*') if p.suffix.lower() in loaders])

for path in files_found:
    try:
        file_sequences = loaders[path.suffix.lower()](path)
        if not file_sequences:
            skipped[f'{path.suffix.lower()} no sequence parsed'] += 1
            continue
        for seq in file_sequences:
            try:
                seq = adapt_joint_count(seq, NUM_JOINTS)
                if seq.shape[0] < MIN_FRAMES:
                    skipped[f'too short < {MIN_FRAMES} frames'] += 1
                    continue
                raw_sequences.append(seq)
            except Exception as e:
                skipped[f'bad shape: {e}'] += 1
    except Exception as e:
        skipped[f'{path.name}: {type(e).__name__}: {e}'] += 1

if not raw_sequences and USE_SYNTHETIC_FALLBACK:
    print('No real compatible data found. Using synthetic fallback for smoke testing only.')
    sequences = [synthetic_mediapipe33(random.randint(130, 220), i % 4) for i in range(1200)]
else:
    sequences = [normalize_mediapipe33(seq) for seq in raw_sequences]

print('Files scanned:', len(files_found))
print('Compatible real sequences:', len(raw_sequences))
print('Training sequences:', len(sequences))
if skipped:
    print('\nSkipped summary:')
    for reason, count in skipped.most_common(20):
        print(f'  {count:4d}  {reason}')

if not sequences:
    raise ValueError(
        'No compatible training sequences found. Upload pose data shaped [T,33,3], [N,T,33,3], '
        '[T,99], long CSV(frame,joint,x,y,z), or wide CSV(x_0,y_0,z_0...).'
    )

lengths = [len(s) for s in sequences]
print('Frame lengths: min=', min(lengths), 'median=', int(np.median(lengths)), 'max=', max(lengths))
print('Example:', sequences[0].shape, sequences[0].dtype)

In [ ]:
seq = sequences[0]
plt.figure(figsize=(8, 4))
plt.plot(seq[:, 15, 0], label='left wrist x')
plt.plot(seq[:, 16, 0], label='right wrist x')
plt.plot(seq[:, 11, 1], label='left shoulder y')
plt.title('Normalized landmark sanity check')
plt.legend()
plt.show()

In [ ]:
KEY_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 31, 32]
GRAPH_EDGES = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24), (23, 25), (25, 27), (24, 26), (26, 28)
]


def compute_motion_features(x):
    # x: torch [B,T,J,C]
    velocity = torch.zeros_like(x)
    velocity[:, 1:] = x[:, 1:] - x[:, :-1]
    acceleration = torch.zeros_like(x)
    acceleration[:, 1:] = velocity[:, 1:] - velocity[:, :-1]
    return velocity, acceleration


def compute_action_context(x):
    # Hand-built orange-style context, computed only from past pose.
    # Returns [B, 8]: motion energy, shoulder/elbow/wrist/knee energies, risk, progress proxy, symmetry.
    velocity, acceleration = compute_motion_features(x)
    speed = torch.linalg.norm(velocity, dim=-1)
    accel = torch.linalg.norm(acceleration, dim=-1)
    motion_energy = speed.mean(dim=(1, 2), keepdim=False)
    shoulder_energy = speed[:, :, [11, 12]].mean(dim=(1, 2))
    elbow_energy = speed[:, :, [13, 14]].mean(dim=(1, 2))
    wrist_energy = speed[:, :, [15, 16]].mean(dim=(1, 2))
    knee_energy = speed[:, :, [25, 26]].mean(dim=(1, 2))
    accel_energy = accel.mean(dim=(1, 2))
    left_wrist = x[:, -1, 15]
    right_wrist = x[:, -1, 16]
    symmetry = torch.linalg.norm(left_wrist - right_wrist, dim=-1)
    progress_proxy = torch.sigmoid((motion_energy - motion_energy.mean()) * 8.0)
    mistake_risk_proxy = torch.sigmoid((accel_energy + symmetry) * 2.0)
    return torch.stack([
        motion_energy,
        shoulder_energy,
        elbow_energy,
        wrist_energy,
        knee_energy,
        accel_energy,
        progress_proxy,
        mistake_risk_proxy,
    ], dim=-1)


def compute_physics_prior(x, future_frames=30):
    velocity, acceleration = compute_motion_features(x)
    last = x[:, -1:]
    v = velocity[:, -1:]
    a = acceleration[:, -1:]
    steps = torch.arange(1, future_frames + 1, device=x.device, dtype=x.dtype).view(1, future_frames, 1, 1)
    dt = steps / future_frames
    return last + v * dt + 0.5 * a * (dt ** 2)


class MotionForecastDataset(Dataset):
    def __init__(self, sequences, past=60, future=30, stride=5, max_windows=50000):
        self.sequences = sequences
        self.items = []
        for si, seq in enumerate(sequences):
            if len(seq) < past + future:
                continue
            for start in range(0, len(seq) - past - future + 1, stride):
                self.items.append((si, start))
                if len(self.items) >= max_windows:
                    break
            if len(self.items) >= max_windows:
                break
        self.past = past
        self.future = future

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        si, start = self.items[idx]
        seq = self.sequences[si]
        x = seq[start:start+self.past]
        y = seq[start+self.past:start+self.past+self.future]
        return torch.from_numpy(x).float(), torch.from_numpy(y).float()


ds = MotionForecastDataset(sequences, PAST_FRAMES, FUTURE_FRAMES, STRIDE, MAX_WINDOWS)
print('Training windows:', len(ds))
if len(ds) < 2:
    raise ValueError('Not enough training windows. Add longer sequences or reduce PAST_FRAMES/FUTURE_FRAMES.')

val_len = max(1, int(len(ds) * 0.15))
train_len = len(ds) - val_len
if train_len < 1:
    train_len, val_len = len(ds) - 1, 1
train_ds, val_ds = random_split(ds, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, train_len), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=min(BATCH_SIZE, val_len), shuffle=False, num_workers=0)
print('Train:', len(train_ds), 'Val:', len(val_ds), 'Batch:', train_loader.batch_size)

In [ ]:
class GraphAttentionBlock(nn.Module):
    def __init__(self, d_model, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        x = self.norm1(x + a)
        return self.norm2(x + self.ff(x))


class ACPSTGATMotionPredictor(nn.Module):
    def __init__(self, num_joints=33, coord_dim=3, d_model=128, heads=4, temporal_layers=3, future_frames=30):
        super().__init__()
        self.num_joints = num_joints
        self.coord_dim = coord_dim
        self.future_frames = future_frames
        self.input_embed = nn.Linear(coord_dim * 3, d_model)  # pose + velocity + acceleration
        self.joint_id = nn.Parameter(torch.randn(1, 1, num_joints, d_model) * 0.02)
        self.graph_attn = GraphAttentionBlock(d_model, heads)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, activation='gelu'
        )
        self.temporal = nn.TransformerEncoder(enc, num_layers=temporal_layers)
        self.action_context = nn.Sequential(
            nn.Linear(8, d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.joint_gate = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, num_joints), nn.Sigmoid()
        )
        self.future_queries = nn.Parameter(torch.randn(1, future_frames, d_model) * 0.02)
        dec = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, activation='gelu'
        )
        self.decoder = nn.TransformerDecoder(dec, num_layers=2)
        self.residual_out = nn.Linear(d_model, num_joints * coord_dim)
        self.physics_blend = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, 1), nn.Sigmoid())

    def forward(self, x):
        B, T, J, C = x.shape
        velocity, acceleration = compute_motion_features(x)
        physics_prior = compute_physics_prior(x, self.future_frames)
        action = compute_action_context(x)
        action_embed = self.action_context(action)

        features = torch.cat([x, velocity, acceleration], dim=-1)
        h = self.input_embed(features) + self.joint_id[:, :, :J, :]
        h = self.graph_attn(h.reshape(B * T, J, -1)).reshape(B, T, J, -1)

        joint_weights = self.joint_gate(action_embed).view(B, 1, J, 1)
        pooled = (h * (0.5 + joint_weights)).mean(dim=2)
        memory = self.temporal(pooled) + action_embed.unsqueeze(1)

        q = self.future_queries.repeat(B, 1, 1) + action_embed.unsqueeze(1)
        decoded = self.decoder(q, memory)
        residual = self.residual_out(decoded).reshape(B, self.future_frames, self.num_joints, self.coord_dim)
        blend = self.physics_blend(action_embed).view(B, 1, 1, 1)
        return physics_prior * blend + (physics_prior + residual) * (1.0 - blend)

model = ACPSTGATMotionPredictor(NUM_JOINTS, COORD_DIM, 128, 4, 3, FUTURE_FRAMES).to(DEVICE)
print('Model: ACP-STGAT')
print('Params M:', round(sum(p.numel() for p in model.parameters()) / 1e6, 3))

In [ ]:
def mpjpe(pred, target):
    return torch.norm(pred - target, dim=-1).mean()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 12
best_val = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch} train'):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred = model(x)
        loss_main = mpjpe(pred, y)
        physics = compute_physics_prior(x, FUTURE_FRAMES)
        loss_physics = F.smooth_l1_loss(pred - physics, y - physics)
        loss = loss_main + 0.15 * loss_physics
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss_main.item() * x.size(0)
    train_loss = total / len(train_loader.dataset)

    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f'Epoch {epoch} val'):
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            total += mpjpe(model(x), y).item() * x.size(0)
    val_loss = total / len(val_loader.dataset)
    history.append((train_loss, val_loss))
    print(f'Epoch {epoch}: train={train_loss:.5f}, val={val_loss:.5f}')

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'model_state': model.state_dict(),
            'model_name': 'ACP-STGAT',
            'past_frames': PAST_FRAMES,
            'future_frames': FUTURE_FRAMES,
            'num_joints': NUM_JOINTS,
            'coord_dim': COORD_DIM,
            'normalization': 'hip_center_origin_shoulder_or_torso_scale',
            'features': ['pose', 'velocity', 'acceleration', 'physics_prior', 'action_context', 'joint_attention']
        }, 'acp_stgat_motion_predictor_mediapipe33.pt')
        print('Saved best checkpoint')

plt.figure(figsize=(7, 4))
plt.plot([h[0] for h in history], label='train')
plt.plot([h[1] for h in history], label='val')
plt.title('ACP-STGAT loss')
plt.legend()
plt.show()

In [ ]:
ckpt = torch.load('acp_stgat_motion_predictor_mediapipe33.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

dummy = torch.randn(1, PAST_FRAMES, NUM_JOINTS, COORD_DIM).to(DEVICE)
onnx_path = 'acp_stgat_motion_predictor.onnx'

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=['past_landmarks'],
    output_names=['future_landmarks'],
    opset_version=18,
    dynamo=False
)

metadata = {
    'model_name': 'ACP-STGAT',
    'display_name': 'Action-Conditioned Physics-Informed ST-GAT',
    'past_frames': PAST_FRAMES,
    'future_frames': FUTURE_FRAMES,
    'num_joints': NUM_JOINTS,
    'coord_dim': COORD_DIM,
    'normalization': 'hip_center_origin_shoulder_or_torso_scale',
    'opset_version': 18,
    'input_names': ['past_landmarks'],
    'output_names': ['future_landmarks'],
    'features_inside_model': ['velocity', 'acceleration', 'physics_prior', 'action_context', 'joint_attention']
}
with open('acp_stgat_motion_predictor_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Exported:', onnx_path)
print(metadata)

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession('acp_stgat_motion_predictor.onnx', providers=['CPUExecutionProvider'])
example = np.random.randn(1, PAST_FRAMES, NUM_JOINTS, COORD_DIM).astype(np.float32)
out = sess.run(None, {'past_landmarks': example})[0]
print('Input:', example.shape)
print('Output:', out.shape)
assert out.shape == (1, FUTURE_FRAMES, NUM_JOINTS, COORD_DIM)

In [ ]:
from google.colab import files
files.download('acp_stgat_motion_predictor.onnx')
files.download('acp_stgat_motion_predictor_metadata.json')
files.download('acp_stgat_motion_predictor_mediapipe33.pt')